# 03 Baselines and classical model benchmark

Day 6: two trivial baselines (the floor). Day 7: four classical models, all inside the leak-free pipeline, evaluated with stratified 5-fold CV on the **training split only**. Validation and test splits are never loaded here.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

In [2]:
import pandas as pd  # noqa: E402
from sklearn.dummy import DummyClassifier  # noqa: E402
from sklearn.ensemble import RandomForestClassifier  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import (  # noqa: E402
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (  # noqa: E402
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
)
from sklearn.neighbors import KNeighborsClassifier  # noqa: E402
from sklearn.tree import DecisionTreeClassifier  # noqa: E402

from src.pipeline import (  # noqa: E402
    CATEGORICAL_FEATURES,
    SEED,
    build_pipeline,
    load_splits,
    split_xy,
)

train, _, _ = load_splits()
X_train, y_train = split_xy(train)
print(X_train.shape, f"anomaly rate = {y_train.mean():.4f}")

(17500, 9) anomaly rate = 0.0391


In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}
baselines = {
    "Majority class": DummyClassifier(strategy="most_frequent"),
    "Stratified dummy": DummyClassifier(strategy="stratified", random_state=SEED),
}

def run_cv(model_dict, x, y, cat_cols=None):
    """Cross-validate each model inside the leak-free pipeline.

    Returns:
        (mean_table, std_table) of every metric over the 5 folds.
    """
    rows = {}
    for name, model in model_dict.items():
        pipe = build_pipeline(model, cat_cols=cat_cols)
        res = cross_validate(pipe, x, y, cv=cv, scoring=scoring)
        rows[name] = {m: res[f"test_{m}"] for m in scoring}

    mean = pd.DataFrame({n: {m: s.mean() for m, s in r.items()} for n, r in rows.items()}).T
    std = pd.DataFrame({n: {m: s.std() for m, s in r.items()} for n, r in rows.items()}).T
    return mean, std

## Day 6: trivial baselines

In [4]:
mean_table, std_table = run_cv(baselines, X_train, y_train)
print("Mean over 5 folds")
print(mean_table.round(4))
print()
print("Std over 5 folds")
print(std_table.round(4))

Mean over 5 folds
                  accuracy  precision  recall      f1  roc_auc  pr_auc
Majority class      0.9609     0.0000   0.000  0.0000   0.5000  0.0391
Stratified dummy    0.9239     0.0514   0.054  0.0527   0.5067  0.0399

Std over 5 folds
                  accuracy  precision  recall      f1  roc_auc  pr_auc
Majority class      0.0000     0.0000  0.0000  0.0000   0.0000  0.0000
Stratified dummy    0.0008     0.0094  0.0099  0.0097   0.0052  0.0007


In [5]:
for name, model in baselines.items():
    pred = cross_val_predict(build_pipeline(model), X_train, y_train, cv=cv)
    print(name)
    print(confusion_matrix(y_train, pred))
    print()

Majority class
[[16815     0]
 [  685     0]]

Stratified dummy
[[16132   683]
 [  648    37]]



Accuracy is misleading here: always predicting "normal" scores about 96% accuracy but has recall 0 and F1 0. Both baselines have ROC-AUC near 0.50 and PR-AUC near 0.04 (the anomaly rate), so that is the floor every real model must beat.

## Day 7: four classical models

Same pipeline, same folds, same metrics as the baselines. Each model is refit inside every fold, so the imputer, scaler and encoder only ever see that fold's training rows.

In [6]:
models = {
    "Logistic Regression (L2)": LogisticRegression(max_iter=1000, random_state=SEED),
    "kNN (k=7)": KNeighborsClassifier(n_neighbors=7),
    "Decision Tree (depth 6)": DecisionTreeClassifier(max_depth=6, random_state=SEED),
    "Random Forest (balanced)": RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=SEED, n_jobs=-1
    ),
}

mean_all, std_all = run_cv(models, X_train, y_train)
print("Mean over 5 folds (all features)")
print(mean_all.round(4))
print()
print("Std over 5 folds (all features)")
print(std_all.round(4))

Mean over 5 folds (all features)
                          accuracy  precision  recall      f1  roc_auc  pr_auc
Logistic Regression (L2)    0.9603     0.3755  0.0263  0.0491   0.8092  0.2221
kNN (k=7)                   0.9606     0.1333  0.0029  0.0057   0.6346  0.0803
Decision Tree (depth 6)     0.9593     0.4163  0.0934  0.1514   0.7782  0.1941
Random Forest (balanced)    0.9540     0.3148  0.1460  0.1990   0.8082  0.2061

Std over 5 folds (all features)
                          accuracy  precision  recall      f1  roc_auc  pr_auc
Logistic Regression (L2)    0.0011     0.2014  0.0176  0.0325   0.0089  0.0374
kNN (k=7)                   0.0003     0.2667  0.0058  0.0114   0.0164  0.0106
Decision Tree (depth 6)     0.0010     0.0358  0.0169  0.0220   0.0234  0.0229
Random Forest (balanced)    0.0018     0.0350  0.0131  0.0171   0.0175  0.0128


In [7]:
for name, model in models.items():
    pred = cross_val_predict(build_pipeline(model), X_train, y_train, cv=cv)
    print(name)
    print(confusion_matrix(y_train, pred))
    print()

Logistic Regression (L2)
[[16788    27]
 [  667    18]]

kNN (k=7)
[[16808     7]
 [  683     2]]

Decision Tree (depth 6)
[[16723    92]
 [  621    64]]

Random Forest (balanced)
[[16595   220]
 [  585   100]]



## Does error_code carry the result?

Week 1 found `error_code` was the strongest signal and may leak the label. Rerun the same four models without it and compare.

In [8]:
cat_no_err = [c for c in CATEGORICAL_FEATURES if c != "error_code"]
mean_noerr, std_noerr = run_cv(models, X_train, y_train, cat_cols=cat_no_err)

floor = mean_table.loc["Stratified dummy"]
print(f"Baseline floor: PR-AUC {floor['pr_auc']:.4f}, ROC-AUC {floor['roc_auc']:.4f}")
print()

compare = pd.DataFrame(
    {
        "PR-AUC with error_code": mean_all["pr_auc"],
        "PR-AUC without": mean_noerr["pr_auc"],
        "ROC-AUC with error_code": mean_all["roc_auc"],
        "ROC-AUC without": mean_noerr["roc_auc"],
    }
)
print(compare.round(4))
print()
print("Beats baseline on PR-AUC and ROC-AUC (all features):")
print(
    ((mean_all["pr_auc"] > floor["pr_auc"]) & (mean_all["roc_auc"] > floor["roc_auc"]))
    .to_dict()
)

Baseline floor: PR-AUC 0.0399, ROC-AUC 0.5067

                          PR-AUC with error_code  PR-AUC without  \
Logistic Regression (L2)                  0.2221          0.0951   
kNN (k=7)                                 0.0803          0.0595   
Decision Tree (depth 6)                   0.1941          0.0946   
Random Forest (balanced)                  0.2061          0.0942   

                          ROC-AUC with error_code  ROC-AUC without  
Logistic Regression (L2)                   0.8092           0.6549  
kNN (k=7)                                  0.6346           0.5803  
Decision Tree (depth 6)                    0.7782           0.6501  
Random Forest (balanced)                   0.8082           0.6484  

Beats baseline on PR-AUC and ROC-AUC (all features):
{'Logistic Regression (L2)': True, 'kNN (k=7)': True, 'Decision Tree (depth 6)': True, 'Random Forest (balanced)': True}


## Reproducibility check

Rerun with the same seeds. Results must match within 0.5%.

In [9]:
mean_again, _ = run_cv(models, X_train, y_train)
print("Max change in any metric on rerun:", float((mean_again - mean_all).abs().max().max()))

Max change in any metric on rerun: 0.0


## Takeaway

Fill in after running: which model has the best PR-AUC, how far it is above the 0.04 floor, and how much PR-AUC drops without `error_code`.